# Лабораторная работа № 5.1
## Структурная идентификация

**Выполнили:**

- Левченков Артемий Викторович;
- Бикбулатова Динара Фануровна;
- Ладыгин Максим Александрович.

**Группа:** БИВТ-24-АЭ-3

**Исходные данные:** дискретный ряд распределения возраста людей, совершивших преступления в Москве в 2021 году, из ЛР 1.

Абсолютные значения округляем до десятых, проценты — до сотых.

$x$ — возраст, $y$ — абсолютная частота, $N=\sum_i y_i$ — общая сумма частот.

In [1]:
import pandas as pd

data = pd.read_table("data/moscow.txt", header=None)[0]
discrete = data.value_counts().sort_index().rename_axis("x").reset_index(name="y")
x = discrete["x"]
y = discrete["y"]
N = sum(y)

print("Общая сумма частот N =", N)

Общая сумма частот N = 32423


### 1. Сравнение семи видов зависимостей

Используем начальную и конечную точки $(x_1,y_1)$ и $(x_n,y_n)$. Средние для каждого вида зависимости:

| № | Зависимость | $\bar x_s$ | $\bar y_s$ |
|---|---|---|---|
| 1 | $y=ax+b$ | $(x_1+x_n)/2$ | $(y_1+y_n)/2$ |
| 2 | $y=ax^b$ | $\sqrt{x_1x_n}$ | $\sqrt{y_1y_n}$ |
| 3 | $y=ab^x$ | $(x_1+x_n)/2$ | $\sqrt{y_1y_n}$ |
| 4 | $y=a+b/x$ | $2x_1x_n/(x_1+x_n)$ | $(y_1+y_n)/2$ |
| 5 | $y=1/(ax+b)$ | $(x_1+x_n)/2$ | $2y_1y_n/(y_1+y_n)$ |
| 6 | $y=x/(ax+b)$ | $2x_1x_n/(x_1+x_n)$ | $2y_1y_n/(y_1+y_n)$ |
| 7 | $y=a\ln x+b$ | $\sqrt{x_1x_n}$ | $(y_1+y_n)/2$ |

In [2]:
x1 = x.iloc[0]
xn = x.iloc[-1]
y1 = y.iloc[0]
yn = y.iloc[-1]

x_a = round((x1 + xn) / 2, 1)
x_g = round((x1 * xn) ** 0.5, 1)
x_h = round(2 * x1 * xn / (x1 + xn), 1)
y_a = round((y1 + yn) / 2, 1)
y_g = round((y1 * yn) ** 0.5, 1)
y_h = round(2 * y1 * yn / (y1 + yn), 1)

#### Экспериментальное значение и отклонения

Для $x_s=\bar x_s$ берём частоту из ряда или находим её линейной интерполяцией:

$$y_s=y_i+\frac{y_{i+1}-y_i}{x_{i+1}-x_i}(x_s-x_i).$$

$$\Delta_s=|\bar y_s-y_s|,\qquad \Delta_s\%=\frac{\Delta_s}{N}\cdot100\%.$$

In [3]:
def experimental_value(xs, x, y):
    for i in range(len(x)):
        if xs == x.iloc[i]:
            return round(float(y.iloc[i]), 1)

    for i in range(len(x) - 1):
        if x.iloc[i] < xs < x.iloc[i + 1]:
            ys = y.iloc[i] + (y.iloc[i + 1] - y.iloc[i]) / (x.iloc[i + 1] - x.iloc[i]) * (xs - x.iloc[i])
            return round(float(ys), 1)


dependencies = [
    ["Линейная: y = ax + b", x_a, y_a],
    ["Степенная: y = ax^b", x_g, y_g],
    ["Показательная: y = ab^x", x_a, y_g],
    ["Гиперболическая: y = a + b/x", x_h, y_a],
    ["y = 1/(ax + b)", x_a, y_h],
    ["y = x/(ax + b)", x_h, y_h],
    ["Логарифмическая: y = a ln(x) + b", x_g, y_a]
]

rows = []
for name, mean_x, mean_y in dependencies:
    ys = experimental_value(mean_x, x, y)
    delta = round(abs(mean_y - ys), 1)
    percent = round(delta / N * 100, 2)
    rows.append([name, mean_x, mean_y, ys, delta, percent])

results = pd.DataFrame(rows, columns=["Вид зависимости", "x_mean", "y_mean", "y_s", "delta", "delta_percent"])
results.index = range(1, 8)
results.index.name = "№"
display(results.style.format({"x_mean": "{:.1f}", "y_mean": "{:.1f}", "y_s": "{:.1f}",
                              "delta": "{:.1f}", "delta_percent": "{:.2f}"}))

best = results.loc[results["delta"].idxmin()]
print("Минимальное отклонение:", best["Вид зависимости"])
print(f"Δ = {best['delta']:.1f}, доля от общей суммы частот = {best['delta_percent']:.2f}%")

,Вид зависимости,x_mean,y_mean,y_s,delta,delta_percent
№,,,,,,
1,Линейная: y = ax + b,43.5,33.0,926.0,893.0,2.75
2,Степенная: y = ax^b,32.0,28.9,796.0,767.1,2.37
3,Показательная: y = ab^x,43.5,28.9,926.0,897.1,2.77
4,Гиперболическая: y = a + b/x,23.5,33.0,940.5,907.5,2.80
5,y = 1/(ax + b),43.5,25.2,926.0,900.8,2.78
6,y = x/(ax + b),23.5,25.2,940.5,915.3,2.82
7,Логарифмическая: y = a ln(x) + b,32.0,33.0,796.0,763.0,2.35


Минимальное отклонение: Логарифмическая: y = a ln(x) + b
Δ = 763.0, доля от общей суммы частот = 2.35%


Логарифмическая зависимость имеет минимальное отклонение среди семи видов, но не описывает немонотонный характер ряда. Для дальнейшей аппроксимации рассматриваем полином.

### 2. Определение степени многочлена

Для шага $\Delta x=1$ частоты в отсутствующих точках 68 и 72 года находим интерполяцией. Сумма частот исходного ряда $N$ остаётся прежней.

In [4]:
rows = []
for age in range(int(x1), int(xn) + 1):
    frequency = experimental_value(age, x, y)
    rows.append([age, frequency])

differences = pd.DataFrame(rows, columns=["x", "y"])

#### Последовательные разности

Вычисляем модули разностей соседних значений:

$$d_i^{(0)}=y_i,\qquad d_i^{(k)}=|d_i^{(k-1)}-d_{i-1}^{(k-1)}|.$$

$$M_k=\max_i d_i^{(k)},\qquad M_k\%=\frac{M_k}{N}\cdot100\%.$$

Выбираем первый порядок $m$, для которого $M_m\le0{,}02N$.

In [5]:
limit = round(0.02 * N, 1)
previous = differences["y"].tolist()
rows = []
degree = None

for order in range(1, len(previous)):
    current = []
    for i in range(1, len(previous)):
        difference = round(abs(previous[i] - previous[i - 1]), 1)
        current.append(difference)

    differences[f"d{order}"] = [None] * order + current
    maximum = max(current)
    percent = round(maximum / N * 100, 2)
    rows.append([order, maximum, percent])

    if maximum <= limit:
        degree = order
        break

    previous = current

summary = pd.DataFrame(rows, columns=["Порядок", "Максимум разностей", "Процент от N"])
print(f"Порог 2% от N = {limit:.1f}")
display(summary.style.format({"Максимум разностей": "{:.1f}", "Процент от N": "{:.2f}"}))

print("Все найденные последовательные разности:")
with pd.option_context("display.max_rows", None):
    display(differences.style.format(precision=1, na_rep="").format({"x": "{:.0f}"}))

print("Степень аппроксимирующего многочлена m =", degree)

Порог 2% от N = 648.5


,Порядок,Максимум разностей,Процент от N
0,1,910.0,2.81
1,2,876.0,2.70
2,3,839.0,2.59
3,4,835.0,2.58
4,5,639.0,1.97


Все найденные последовательные разности:


,x,y,d1,d2,d3,d4,d5
0,14,49.000000,nan,nan,nan,nan,nan
1,15,53.000000,4.000000,nan,nan,nan,nan
2,16,236.000000,183.000000,179.000000,nan,nan,nan
3,17,126.000000,110.000000,73.000000,106.000000,nan,nan
4,18,806.000000,680.000000,570.000000,497.000000,391.000000,nan
5,19,971.000000,165.000000,515.000000,55.000000,442.000000,51.000000
6,20,910.000000,61.000000,104.000000,411.000000,356.000000,86.000000
7,21,716.000000,194.000000,133.000000,29.000000,382.000000,26.000000
8,22,944.000000,228.000000,34.000000,99.000000,70.000000,312.000000
9,23,1071.000000,127.000000,101.000000,67.000000,32.000000,38.000000


Степень аппроксимирующего многочлена m = 5
